# Machine Learning Consensus Phenotyping (ML-consensus)

This notebook implements the same consensus-driven ML workflow exposed by the SPACEMAP GUI. Agreement among automated phenotyping methods creates pseudo-labels; manual-reference columns are never allowed to participate in that vote.

## Rationale

Cells with a unique winning phenotype supported by at least `MIN_CONSENSUS_METHODS` become consensus pseudo-labels. Ties such as 2-vs-2 are treated as non-consensus, and catch-all labels configured in `EXCLUDED_PHENOTYPES` are ignored as votes. A supervised classifier learns from permitted consensus cells and predicts non-consensus cells.

When `VALIDATION_GROUND_TRUTH_COLUMN` is provided, every manually labeled row in that column is excluded from pseudo-label model training. The final complete Consensus-ML output—direct consensus labels for consensus cells and model predictions for non-consensus cells—is then evaluated against those held-out manual labels. Without a manual validation column, the internal split is exploratory and measures reproduction of pseudo-labels rather than biological ground truth.

## Outputs

- `phenotype_ml_consensus`: final phenotype.
- `ml_consensus_score`: agreeing-method fraction for direct consensus assignments; maximum model probability for model-imputed cells. This is an uncertainty score, not a calibrated probability.
- `consensus`: binary direct-consensus flag.
- `phenotype_ml_consensus_refined`: low-confidence predictions relabeled according to the configured threshold.

Artifacts are saved under `outputs/phenotyping/ML_consensus`.


## Step 1: Configuration

Configure input paths, consensus parameters, and model hyperparameters. Adjust `MIN_CONSENSUS_METHODS` to control strictness (e.g. 3 out of 4 methods).

## N_ESTIMATORS Optimization Notes

**Current setting: 500 trees**
- Good balance between accuracy and training time
- Suitable for datasets with ~20 markers and multiple cell types
- Consider testing different values:
  - **300**: Faster training, may sacrifice some accuracy
  - **500**: Current balanced choice
  - **1000**: Potentially better accuracy, longer training time
  - **1500+**: Diminishing returns likely

**To test optimal value**: Run the analysis with different N_ESTIMATORS values and compare the validation accuracy in the classification report. Look for the point where accuracy plateaus.

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')

project_root = Path().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
# Use ProjectConfig for centralized path management
from spacemap.config import ProjectConfig, setup_logging, log_step
ACTIVE_PROJECT_DIR = project_root / 'project' / 'INNATE_4_IMAGES_07302026'
config = ProjectConfig(ACTIVE_PROJECT_DIR)
logger = setup_logging(__name__)

from spacemap.ml_consensus import ConsensusConfig, run_ml_consensus_analysis

UNIFIED_CSV = config.output_dir / 'batch_corrected' / 'batch_corrected_merged_with_ref_enriched.csv'
OUTPUT_DIR = config.output_dir / 'phenotyping' / 'ML_consensus'

PHENOTYPING_METHODS = [
    'phenotype_leiden',
    'phenotype_som',
    'phenotype_threshold',
    'phenotype_vortex'
]
EXCLUDED_PHENOTYPES = ['Others']  # Ignore catch-all labels when constructing pseudo-labels

# Optional held-out manual reference. These rows are excluded from pseudo-label model training.
VALIDATION_GROUND_TRUTH_COLUMN = 'phenotype_ref1b'
ANALYSIS_FILTER_COLUMN = 'tissue'  # Restrict fitting/validation/prediction to eligible tissue cells
ANALYSIS_FILTER_VALUE = 1
DROP_INVALID_MARKER_ROWS = True  # Match the GUI complete-marker eligibility check
WRITE_BACKUP_CSV = False  # Review-first default: do not create an automatic result CSV
UPDATE_SOURCE_CSV = False  # Save explicitly only after reviewing assignments

# Marker configuration (upstream batch-corrected intensities; extra Z-normalization is optional)
CORR_MARKERS = [    "CD31", "E-cadherin", "CD20", "CD4", "Pan-Cytokeratin", "CD68", "Podoplanin", "CD11c", "CD8", "HLA-DR",
    "CD3e", "CD45", "CD15", "FoxP3", "PDGFRa", "aSMA", "CD163", "FN1", "S100A9", "CD11b", "CD38",
    "Vimentin"]

NON_CORR_MARKERS = [
]

# Combined marker list for clustering
MARKERS_BASE = CORR_MARKERS + NON_CORR_MARKERS
CLUSTERING_MARKERS_CORR = [f"{m}_corr" for m in CORR_MARKERS] + NON_CORR_MARKERS

# New: Morphology features (transformed: log1p for sizes/lengths, sin/cos for orientation; then Z-norm if enabled)
# Purpose: Incorporate cell shape/context for better phenotype discrimination.
# Usage: List exact column names from unified CSV; set to [] to disable.
MORPHOLOGY_COLS = []

# New: Compartment binaries (0/1 indicators; optionally down-weighted for soft context)
# Purpose: Provide spatial priors without hard-gating, preserving rare infiltrates.
# Usage: List exact binary column names; set to [] to disable.
COMPARTMENT_COLS = []  # Adjust to your column names

# Feature-block controls. Set a block to 0.0 to exclude it. With standardized/tree
# classifiers, nonzero values should not be interpreted as learned feature importance.
FEATURE_WEIGHTS = {'markers': 1.0, 'morphology': 0.0, 'compartments': 0.0}

# Z-normalization parameters (applies to markers and transformed morphology; skips binaries)
# The selected corrected markers already passed upstream batch correction.
USE_Z_NORMALIZATION = False
BATCH_NORMALIZATION_COL = 'batch_id'

MIN_CONSENSUS_METHODS = 3
TEST_SIZE = 0.30
N_ESTIMATORS = 1500
RANDOM_STATE = 42
ML_METHOD = 'random_forest'
MIN_TRAINING_CELLS = 50
FIGURE_DPI = 300

# --- Output Column Configuration ---
# Column name for the primary ML consensus phenotype assignment
OUTPUT_PHENOTYPE_COL = 'phenotype_ml_consensus'
# Assignment score: agreement fraction for consensus cells; max class probability for imputed cells
CONFIDENCE_COL_NAME = 'ml_consensus_score'
# Binary column: 1 = consensus cell, 0 = non-consensus cell
CONSENSUS_COL_NAME = 'consensus'

# --- Refined Phenotype Column ---
# Cells with confidence below this threshold are relabeled in the refined column
REFINED_CONFIDENCE_THRESHOLD = 0.75
# Label to assign to low-confidence cells (e.g., 'Others', 'Unclassified')
REFINED_LOW_CONFIDENCE_LABEL = 'Others'
# Refined column will be auto-named: f'{OUTPUT_PHENOTYPE_COL}_refined'

print('ML Consensus Configuration:')
print(f'- Unified CSV: {UNIFIED_CSV}')
print(f'- Output directory: {OUTPUT_DIR}')
print(f'- Methods: {PHENOTYPING_METHODS}')
print(f'- Labels excluded from consensus voting: {EXCLUDED_PHENOTYPES}')
print(f'- Held-out validation: {VALIDATION_GROUND_TRUTH_COLUMN or "internal pseudo-label split"}')
print(f'- Analysis population: {ANALYSIS_FILTER_COLUMN} == {ANALYSIS_FILTER_VALUE!r}')
print(f'- Drop rows with invalid selected markers: {DROP_INVALID_MARKER_ROWS}')
print(f'- Automatic result CSV: {WRITE_BACKUP_CSV}')
print(f'- Update source CSV during run: {UPDATE_SOURCE_CSV}')
print(f'- Selected markers: {len(CLUSTERING_MARKERS_CORR)} total')
print(f'  * Corrected markers: {len(CORR_MARKERS)}')
print(f'  * Non-corrected markers: {len(NON_CORR_MARKERS)}')
print(f'- Morphology features: {MORPHOLOGY_COLS} (transformed and Z-norm if enabled)')
print(f'- Compartment binaries: {COMPARTMENT_COLS} (weighted as soft context)')
print(f'- Feature weights: {FEATURE_WEIGHTS}')
print(f'- Z-normalization: {USE_Z_NORMALIZATION} (grouped by "{BATCH_NORMALIZATION_COL}")')
print(f'- Min consensus methods: {MIN_CONSENSUS_METHODS}')
print(f'- Classifier: {ML_METHOD}, estimators: {N_ESTIMATORS}, internal test size: {TEST_SIZE}')
print(f'- Min training cells: {MIN_TRAINING_CELLS}')
print(f'- Output columns:')
print(f'  * Phenotype: {OUTPUT_PHENOTYPE_COL}')
print(f'  * Confidence: {CONFIDENCE_COL_NAME}')
print(f'  * Consensus binary: {CONSENSUS_COL_NAME}')
print(f'  * Refined: {OUTPUT_PHENOTYPE_COL}_refined (threshold={REFINED_CONFIDENCE_THRESHOLD}, label="{REFINED_LOW_CONFIDENCE_LABEL}")')


## Step 2: Data Integrity Checks
Inspect the unified CSV to confirm required columns and availability of marker features and potential consensus cells.

In [ ]:
import pandas as pd, numpy as np
from spacemap.ml_consensus import identify_consensus

if not UNIFIED_CSV.exists():
    raise FileNotFoundError(f'Unified CSV not found: {UNIFIED_CSV}')
source_preview = pd.read_csv(UNIFIED_CSV)
if ANALYSIS_FILTER_COLUMN not in source_preview.columns:
    raise KeyError(f'Analysis filter column not found: {ANALYSIS_FILTER_COLUMN}')
analysis_mask = source_preview[ANALYSIS_FILTER_COLUMN].astype('string').str.strip().eq(str(ANALYSIS_FILTER_VALUE))
analysis_preview = source_preview.loc[analysis_mask].copy()
if analysis_preview.empty:
    raise ValueError('The configured analysis filter selected no rows')
existing_markers = [column for column in CLUSTERING_MARKERS_CORR if column in analysis_preview.columns]
missing_markers = [column for column in CLUSTERING_MARKERS_CORR if column not in analysis_preview.columns]
if missing_markers:
    raise KeyError(f'Missing selected markers: {missing_markers}')
if DROP_INVALID_MARKER_ROWS:
    marker_values = analysis_preview[existing_markers].apply(pd.to_numeric, errors='coerce')
    finite_marker_mask = np.isfinite(marker_values.to_numpy()).all(axis=1)
    dropped = int((~finite_marker_mask).sum())
    analysis_preview = analysis_preview.loc[finite_marker_mask].copy()
    print(f'Complete-marker population: {len(analysis_preview):,} rows ({dropped:,} excluded)')
preview = analysis_preview.head(5000).copy()

missing_methods = [method for method in PHENOTYPING_METHODS if method not in preview.columns]
if missing_methods:
    raise KeyError(f'Missing phenotyping methods: {missing_methods}')
reference_like_methods = [
    method for method in PHENOTYPING_METHODS
    if method.startswith('phenotype_ref') or method in {'phenotype_bd', 'phenotype_ap'}
]
if reference_like_methods:
    raise ValueError(f'Manual references cannot be consensus methods: {reference_like_methods}')
if not 2 <= MIN_CONSENSUS_METHODS <= len(PHENOTYPING_METHODS):
    raise ValueError('MIN_CONSENSUS_METHODS must be between 2 and the number of selected methods')

print(f'Analysis population: {len(analysis_preview):,}/{len(source_preview):,} rows')
print(f'Consensus preview rows: {len(preview):,}')
print(f'Automated consensus methods: {PHENOTYPING_METHODS}')
print(f'Selected markers found: {len(existing_markers)}/{len(CLUSTERING_MARKERS_CORR)}')
if VALIDATION_GROUND_TRUTH_COLUMN:
    if VALIDATION_GROUND_TRUTH_COLUMN not in analysis_preview.columns:
        raise KeyError(f'Validation reference not found: {VALIDATION_GROUND_TRUTH_COLUMN}')
    values = analysis_preview[VALIDATION_GROUND_TRUTH_COLUMN].astype('string').str.strip()
    valid = analysis_preview[VALIDATION_GROUND_TRUTH_COLUMN].notna() & ~values.str.lower().isin(
        {'', 'unclassified', 'unknown', 'nan', 'none'}
    )
    print(f'Usable held-out manual labels in analysis population: {int(valid.sum()):,}')

preview_consensus = identify_consensus(
    preview,
    PHENOTYPING_METHODS,
    MIN_CONSENSUS_METHODS,
    EXCLUDED_PHENOTYPES,
)
print(f"Unique-winner consensus fraction in preview: {preview_consensus['is_consensus'].mean():.2%}")
if USE_Z_NORMALIZATION:
    if BATCH_NORMALIZATION_COL not in preview.columns:
        raise KeyError(f'Batch normalization column not found: {BATCH_NORMALIZATION_COL}')
    print(f'Additional batch-wise Z-normalization enabled: {BATCH_NORMALIZATION_COL}')
else:
    print('Additional Z-normalization disabled; using upstream batch-corrected markers.')


## Step 3: Execute ML Consensus Analysis
Run the pipeline to generate ML consensus phenotypes and artifacts.

In [ ]:
config = ConsensusConfig(
    unified_csv=UNIFIED_CSV,
    output_dir=OUTPUT_DIR,
    phenotyping_methods=PHENOTYPING_METHODS,
    excluded_phenotypes=EXCLUDED_PHENOTYPES,
    clustering_markers=CLUSTERING_MARKERS_CORR,
    morphology_cols=MORPHOLOGY_COLS,
    compartment_cols=COMPARTMENT_COLS,
    feature_weights=FEATURE_WEIGHTS,
    use_z_normalization=USE_Z_NORMALIZATION,
    batch_normalization_col=BATCH_NORMALIZATION_COL,
    min_consensus_methods=MIN_CONSENSUS_METHODS,
    test_size=TEST_SIZE,
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE,
    min_training_cells=MIN_TRAINING_CELLS,
    figure_dpi=FIGURE_DPI,
    output_phenotype_col=OUTPUT_PHENOTYPE_COL,
    confidence_col_name=CONFIDENCE_COL_NAME,
    consensus_col_name=CONSENSUS_COL_NAME,
    refined_confidence_threshold=REFINED_CONFIDENCE_THRESHOLD,
    refined_low_confidence_label=REFINED_LOW_CONFIDENCE_LABEL,
    ml_method=ML_METHOD,
    validation_ground_truth_column=VALIDATION_GROUND_TRUTH_COLUMN,
    analysis_filter_column=ANALYSIS_FILTER_COLUMN,
    analysis_filter_value=ANALYSIS_FILTER_VALUE,
    drop_invalid_marker_rows=DROP_INVALID_MARKER_ROWS,
    write_backup_csv=WRITE_BACKUP_CSV,
    update_source_csv=UPDATE_SOURCE_CSV,
)

RESULT_HOLDER = {}
def capture_results(result_frame):
    RESULT_HOLDER['dataframe'] = result_frame

result = run_ml_consensus_analysis(config, results_callback=capture_results)
print('Run status:', 'Success' if result else 'Failure')


## Step 4: Post-Run Validation
Verify new columns and basic distribution / confidence statistics.

In [ ]:
udf = RESULT_HOLDER.get('dataframe')
if udf is None:
    raise RuntimeError('No in-memory Consensus-ML results were returned')
required_cols = [OUTPUT_PHENOTYPE_COL, CONFIDENCE_COL_NAME, CONSENSUS_COL_NAME, f'{OUTPUT_PHENOTYPE_COL}_refined']
missing = [c for c in required_cols if c not in udf.columns]
if missing:
    raise RuntimeError(f'Missing expected columns: {missing}')
result_mask = (
    udf[ANALYSIS_FILTER_COLUMN].astype('string').str.strip().eq(str(ANALYSIS_FILTER_VALUE))
    & udf[OUTPUT_PHENOTYPE_COL].notna()
)
analysis_results = udf.loc[result_mask].copy()
print('In-memory review columns confirmed; source CSV remains unchanged.')
print('Top phenotype distribution:')
print(analysis_results[OUTPUT_PHENOTYPE_COL].value_counts().head(15))
print(f'\nConsensus column ({CONSENSUS_COL_NAME}) distribution:')
print(analysis_results[CONSENSUS_COL_NAME].value_counts())
print(f'\nRefined phenotype ({OUTPUT_PHENOTYPE_COL}_refined) distribution:')
print(analysis_results[f'{OUTPUT_PHENOTYPE_COL}_refined'].value_counts().head(15))
print(f'\nAssignment-score summary (not a calibrated probability):')
print(analysis_results[CONFIDENCE_COL_NAME].describe())
non_cons = analysis_results.loc[analysis_results[CONSENSUS_COL_NAME] == 0, CONFIDENCE_COL_NAME].dropna()
if not non_cons.empty:
    print('\nNon-consensus prediction confidence summary:')
    print(non_cons.describe())
else:
    print('All cells were consensus (no predictions).')
if VALIDATION_GROUND_TRUTH_COLUMN:
    values = analysis_results[VALIDATION_GROUND_TRUTH_COLUMN].astype('string').str.strip()
    valid = analysis_results[VALIDATION_GROUND_TRUTH_COLUMN].notna() & ~values.str.lower().isin(
        {'', 'unclassified', 'unknown', 'nan', 'none'}
    )
    held_out = analysis_results.loc[valid]
    accuracy = (held_out[VALIDATION_GROUND_TRUTH_COLUMN] == held_out[OUTPUT_PHENOTYPE_COL]).mean()
    print(f'\nHeld-out final-output accuracy on {VALIDATION_GROUND_TRUTH_COLUMN}: {accuracy:.4f} ({len(held_out):,} cells)')
    print(f'Detailed balanced accuracy, F1 scores, and confusion matrix: {OUTPUT_DIR}/ml_consensus_classification_report.txt')


In [ ]:
#Assigning threshold and unclassified
#import pandas as pd
#import numpy as np

#print(f"Loading unified data from: {UNIFIED_CSV}")
#df = pd.read_csv(UNIFIED_CSV)

# 1. Create the 'phenotype_consensus_refined' column
#print("Creating 'phenotype_consensus_refined' column...")
#df['phenotype_consensus_refined'] = df['phenotype_ml_consensus']
#low_confidence_mask = df['ml_consensus_score'] < 0.5
#df.loc[low_confidence_mask, 'phenotype_consensus_refined'] = 'unclassified'
#print(f"Re-classified {low_confidence_mask.sum()} cells as 'unclassified' due to low confidence.")

# 2. Vectorized consensus calculation
#print("\nAdding 'consensus' and 'non_consensus' flags...")
#pheno_df = df[PHENOTYPING_METHODS].copy()
# Replace non-informative values with NaN
#invalid_phenos = ['unclassified', 'unknown', '']
#for col in PHENOTYPING_METHODS:
#    pheno_df[col] = pheno_df[col].replace(invalid_phenos, np.nan)

# Get the mode (most frequent phenotype) and its count for each row
#modes = pheno_df.mode(axis=1)
#mode_counts = pheno_df.apply(lambda row: row.value_counts().max() if row.count() > 0 else 0, axis=1)

# Determine consensus based on the count of the mode
#is_consensus = (mode_counts >= MIN_CONSENSUS_METHODS) & modes[0].notna()
#df['consensus'] = np.where(is_consensus, 'yes', 'no')
#df['non_consensus'] = np.where(is_consensus, 'no', 'yes')


# 4. Display value counts to verify the new columns
#print("\nValue counts for 'phenotype_consensus_refined':")
#print(df['phenotype_consensus_refined'].value_counts())

#print("\nValue counts for 'consensus':")
#print(df['consensus'].value_counts())

#print("\nValue counts for 'non_consensus':")
#print(df['non_consensus'].value_counts())
# 5. Save the updated DataFrame back to the unified CSV
#print(f"\nSaving updated data back to: {UNIFIED_CSV}")
#df.to_csv(UNIFIED_CSV, index=False)
#print("Save complete.")